[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/07.%20The%20Static%20Berth%20Allocation%20Problem/P7-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 7. The Berth Allocation Problem

## Tier 1 — Mixed-Integer Programming Formulation and Greedy Schedule

The **Berth Allocation Problem (BAP)** assigns ships to berths and chooses when their processing begins. This notebook introduces a mixed-integer programming (MIP) formulation and uses a simple greedy heuristic to construct and visualize a schedule for a small instance.

The distinction between the two is important: the MIP formulation defines an optimization model, but this notebook does not pass that model to an optimization solver. The executable schedule is produced by the greedy heuristic, so it is feasible for the stated instance but is **not guaranteed to be optimal**.

### Learning goals

- Understand how berth assignments and start times create spatial and temporal decisions.
- Formulate ship assignment, arrival-time, and berth non-overlap requirements mathematically.
- Follow how a simple arrival-ordered greedy rule constructs a feasible schedule.
- Interpret a Gantt chart and examine how the greedy schedule changes when fewer berths are available.

### Inputs and outputs

The example uses six ships, three berths, arrival times, processing times, preferred berths, ship sizes, and berth attributes. The notebook produces the generated instance, assignment costs, a greedy schedule, a schedule visualization, and descriptive sensitivity results. It does not produce an optimal MIP solution or solver performance metrics.

### Scope and assumptions

Each ship is assigned to its preferred berth by the greedy rule; the rule does not compare other berths when making that assignment. Berths are treated as interchangeable for feasibility: berth length, depth, and efficiency are descriptive attributes and do not impose physical compatibility or alter processing times. Assignment costs represent preference and a simple size-related penalty. Waiting time and assignment cost are added with equal numerical weight in the stated MIP objective, so the model assumes these quantities have been expressed on a compatible scale. Arrival and processing times are deterministic, and no setup times, tides, channel restrictions, or other operational constraints are modeled.

In [ ]:
# Import the libraries used for data generation, tabulation, and visualization.
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
except ImportError as error:
    raise ImportError(
        "A required dependency is missing. Install numpy, pandas, and matplotlib "
        "in the notebook environment before running this notebook."
    ) from error

print("All required dependencies are available.")

## Problem Definition and MIP Formulation

A berth is a resource that can process at most one ship at a time. Each ship becomes available at its arrival time and occupies its assigned berth for its processing duration. The optimization problem must therefore choose both a berth and a start time for every ship.

### Sets and parameters

- $S = \{1,2,\ldots,n\}$ is the set of ships.
- $B = \{1,2,\ldots,m\}$ is the set of berths.
- $a_i$ is the arrival time of ship $i$.
- $p_i$ is the processing time of ship $i$.
- $c_{ij}$ is the assignment cost when ship $i$ uses berth $j$.
- $M$ is a sufficiently large positive constant used to deactivate a precedence constraint when the corresponding ships are not both assigned to the berth.

The MIP decides which berth each ship uses, when its service begins, and which of two ships comes first when they share a berth. Let $x_{ij}\in\{0,1\}$ equal 1 if ship $i$ is assigned to berth $j$, and let $s_i$ be its start time. For each pair $i<k$, let $y_{ik}\in\{0,1\}$ equal 1 when ship $i$ precedes ship $k$ if they use the same berth. The pair is considered once, with $i<k$, so only one binary ordering decision is needed.

### Objective

The objective combines total waiting time with total assignment cost:

$$\min \quad \sum_{i\in S}(s_i-a_i)+\sum_{i\in S}\sum_{j\in B}c_{ij}x_{ij}.$$

Here, $s_i-a_i$ is ship $i$'s waiting time. The model assigns equal numerical weight to waiting and assignment cost; meaningful use of this sum requires both terms to be on a compatible scale. The objective does not separately maximize berth utilization.

### Constraints

Every ship must use exactly one berth, and service cannot start before the ship arrives:

$$\sum_{j\in B}x_{ij}=1 \quad (i\in S), \qquad s_i\geq a_i \quad (i\in S).$$

For every pair $i<k$ and berth $j$, the following conditional constraints prevent overlap if both ships are assigned to that berth:

$$\begin{aligned}
s_i+p_i &\leq s_k+M(1-y_{ik})+M(2-x_{ij}-x_{kj}),\\
s_k+p_k &\leq s_i+My_{ik}+M(2-x_{ij}-x_{kj}).
\end{aligned}$$

If both ships use berth $j$, then $x_{ij}=x_{kj}=1$ and the assignment-dependent relaxation is zero. If $y_{ik}=1$, the first inequality requires ship $i$ to finish before ship $k$ starts; if $y_{ik}=0$, the second requires the reverse. When the ships do not both use berth $j$, the Big-$M$ terms deactivate the relevant ordering restrictions. A valid implementation must choose $M$ large enough to relax those inequalities over the modeled time range; an excessively large value can weaken numerical performance in a solver.

This pairwise formulation uses $O(n^2m)$ conditional non-overlap constraints. The code below presents the MIP model mathematically; it does not instantiate or solve these variables and constraints. The implemented greedy rule is described separately so that its schedule is not mistaken for a MIP optimum.

In [ ]:
# Generate a reproducible small instance for the scheduling demonstration.
np.random.seed(42)

# Problem parameters
num_ships = 6
num_berths = 3
planning_horizon = 24  # hours used when reporting workload relative to capacity

# Generate ship data
ships = []
for i in range(num_ships):
    ship = {
        'id': i + 1,
        'name': f'Ship_{chr(65 + i)}',
        'arrival_time': np.random.uniform(0, 12),
        'processing_time': np.random.uniform(2, 6),
        'preferred_berth': np.random.randint(1, num_berths + 1),
        'size': np.random.choice(['Small', 'Medium', 'Large'], p=[0.3, 0.4, 0.3])
    }
    ships.append(ship)

# Generate descriptive berth attributes. They do not impose feasibility constraints.
berths = []
for j in range(num_berths):
    berth = {
        'id': j + 1,
        'name': f'Berth_{j + 1}',
        'length': np.random.uniform(200, 400),
        'depth': np.random.uniform(10, 20),
        'efficiency': np.random.uniform(0.8, 1.0)
    }
    berths.append(berth)

# Convert berth preference and the stated size penalties into assignment costs.
def calculate_assignment_cost(ship, berth):
    """Return the nonnegative preference and size-related cost for an assignment."""
    base_cost = 0

    if ship['preferred_berth'] == berth['id']:
        base_cost -= 5
    else:
        base_cost += abs(ship['preferred_berth'] - berth['id']) * 2

    if ship['size'] == 'Large' and berth['length'] < 300:
        base_cost += 10
    elif ship['size'] == 'Small' and berth['length'] > 350:
        base_cost += 3

    return max(0, base_cost)

# The cost matrix is indexed by ship row and berth column, matching the MIP parameter c_ij.
cost_matrix = np.zeros((num_ships, num_berths))
for i, ship in enumerate(ships):
    for j, berth in enumerate(berths):
        cost_matrix[i, j] = calculate_assignment_cost(ship, berth)

print("Generated berth allocation instance")
print(f"Ships: {num_ships}; berths: {num_berths}")

print("\nShip details:")
ship_df = pd.DataFrame(ships)
print(ship_df[['id', 'name', 'arrival_time', 'processing_time', 'preferred_berth', 'size']].round(2))

print("\nBerth details:")
berth_df = pd.DataFrame(berths)
print(berth_df[['id', 'name', 'length', 'depth', 'efficiency']].round(2))

print("\nAssignment cost matrix:")
cost_df = pd.DataFrame(
    cost_matrix,
    index=[ship['name'] for ship in ships],
    columns=[berth['name'] for berth in berths]
)
print(cost_df.round(1))

## Greedy Scheduling Rule

A **greedy heuristic** makes a locally simple choice at each step rather than optimizing all assignment and timing decisions together. Here, ships are processed in increasing order of arrival time and each is assigned to its preferred berth. Its start time is the later of its arrival time and the end of the last scheduled ship at that berth.

For ship $i$ at its preferred berth $j(i)$, the rule is

$$s_i^{\mathrm{greedy}}=\max\left(a_i,\ e_{\mathrm{last}(j(i))}\right), \qquad e_i=s_i^{\mathrm{greedy}}+p_i,$$

where $e_{\mathrm{last}(j(i))}$ is the completion time of the last ship already placed at that berth. If no ship has been placed there, the berth imposes no earlier completion time, so the ship can start at arrival. Sorting by arrival ensures that the last scheduled ship is also the most recently assigned one at that berth; its completion time therefore determines the berth's next available time.

This rule constructs a non-overlapping schedule and respects arrival times, but it does not compare alternative berths or reorder ships to improve a global objective. Its schedule and metrics are consequently heuristic results, not a certificate of MIP optimality.

In [ ]:
def simple_greedy_allocation(ships, berths, cost_matrix):
    """Schedule ships by arrival order at their preferred berths."""
    allocations = []
    berth_schedules = {berth['id']: [] for berth in berths}
    sorted_ships = sorted(ships, key=lambda ship: ship['arrival_time'])

    for ship in sorted_ships:
        preferred_berth = ship['preferred_berth']
        if preferred_berth not in berth_schedules:
            raise ValueError(
                f"Preferred berth {preferred_berth} is not available for {ship['name']}."
            )

        # The berth's final scheduled completion determines its next available time.
        berth_schedule = berth_schedules[preferred_berth]
        earliest_start = ship['arrival_time']
        if berth_schedule:
            last_ship = max(berth_schedule, key=lambda allocation: allocation['end_time'])
            earliest_start = max(earliest_start, last_ship['end_time'])

        allocation = {
            'ship_id': ship['id'],
            'ship_name': ship['name'],
            'berth_id': preferred_berth,
            'start_time': earliest_start,
            'end_time': earliest_start + ship['processing_time'],
            'waiting_time': earliest_start - ship['arrival_time'],
            'cost': cost_matrix[ship['id'] - 1, preferred_berth - 1]
        }
        allocations.append(allocation)
        berth_schedule.append(allocation)

    return allocations, berth_schedules


# Construct the schedule and check its assignment, arrival, and non-overlap properties.
greedy_allocations, greedy_schedules = simple_greedy_allocation(ships, berths, cost_matrix)
assert len(greedy_allocations) == len(ships)
assert {allocation['ship_id'] for allocation in greedy_allocations} == {
    ship['id'] for ship in ships
}

ship_by_id = {ship['id']: ship for ship in ships}
for allocation in greedy_allocations:
    ship = ship_by_id[allocation['ship_id']]
    assert allocation['start_time'] >= ship['arrival_time']
    assert np.isclose(
        allocation['end_time'] - allocation['start_time'],
        ship['processing_time']
    )

for berth_schedule in greedy_schedules.values():
    ordered_schedule = sorted(berth_schedule, key=lambda allocation: allocation['start_time'])
    for previous, following in zip(ordered_schedule, ordered_schedule[1:]):
        assert previous['end_time'] <= following['start_time']

greedy_df = pd.DataFrame(greedy_allocations)
total_waiting = sum(allocation['waiting_time'] for allocation in greedy_allocations)
total_cost = sum(allocation['cost'] for allocation in greedy_allocations)
average_waiting = total_waiting / len(greedy_allocations)

print("Greedy heuristic schedule:")
print(greedy_df[['ship_name', 'berth_id', 'start_time', 'end_time', 'waiting_time', 'cost']].round(2))
print("\nSchedule metrics:")
print(f"Total waiting time: {total_waiting:.2f} hours")
print(f"Average waiting time: {average_waiting:.2f} hours per ship")
print(f"Total assignment cost: {total_cost:.2f}")

## Visualizing the Greedy Schedule

A Gantt chart places time on the horizontal axis and berths on the vertical axis. Each solid bar shows a ship's processing interval; a red outlined bar, when present, shows the interval between arrival and service start. This makes waiting and berth occupancy visible without changing the scheduling rule.

In [ ]:
def create_gantt_chart(allocations, ships, title="Berth Allocation Schedule"):
    """Draw processing and waiting intervals for a berth schedule."""
    fig, ax = plt.subplots(figsize=(14, 6))
    ship_ids = sorted(ship['id'] for ship in ships)
    color_map = dict(zip(ship_ids, plt.cm.Set3(np.linspace(0, 1, len(ship_ids)))))
    ship_by_id = {ship['id']: ship for ship in ships}
    berth_ids = sorted({allocation['berth_id'] for allocation in allocations})
    berth_to_y = {berth_id: position for position, berth_id in enumerate(berth_ids)}
    bar_height = 0.65

    for allocation in allocations:
        ship = ship_by_id[allocation['ship_id']]
        y_position = berth_to_y[allocation['berth_id']]
        start = allocation['start_time']
        duration = allocation['end_time'] - start

        # Processing bars show the interval during which each berth is occupied.
        processing_bar = patches.Rectangle(
            (start, y_position - bar_height / 2),
            duration,
            bar_height,
            linewidth=1,
            edgecolor='black',
            facecolor=color_map[allocation['ship_id']],
            alpha=0.85
        )
        ax.add_patch(processing_bar)
        ax.text(
            start + duration / 2,
            y_position,
            allocation['ship_name'],
            ha='center',
            va='center',
            fontsize=9
        )

        if allocation['waiting_time'] > 0:
            waiting_bar = patches.Rectangle(
                (ship['arrival_time'], y_position - bar_height / 2),
                allocation['waiting_time'],
                bar_height,
                linewidth=1,
                edgecolor='red',
                facecolor='none',
                linestyle='--'
            )
            ax.add_patch(waiting_bar)

    ship_legend = [
        patches.Patch(facecolor=color_map[ship_id], edgecolor='black', label=ship_by_id[ship_id]['name'])
        for ship_id in ship_ids
    ]
    waiting_legend = patches.Patch(
        facecolor='none', edgecolor='red', linestyle='--', label='Waiting time'
    )
    ax.legend(handles=ship_legend + [waiting_legend], loc='upper left', bbox_to_anchor=(1.02, 1))

    ax.set_yticks(range(len(berth_ids)))
    ax.set_yticklabels([f"Berth {berth_id}" for berth_id in berth_ids])
    ax.set_ylim(-0.7, len(berth_ids) - 0.3)
    ax.set_xlim(left=0)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Berth')
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()
    return fig, ax


fig, ax = create_gantt_chart(
    greedy_allocations,
    ships,
    title="Greedy Heuristic — Berth Allocation Schedule"
)
plt.show()

print("Each colored bar represents ship processing; red outlined intervals represent waiting time.")

## Sensitivity to the Number of Available Berths

This experiment reruns the same greedy scheduling rule with prefixes of the generated berth list: first one berth, then two, then all available berths. If a ship's preferred berth is not in the available subset, its preference is mapped to the last available berth for this experiment. Assignment costs are recalculated using that adjusted preference, so the costs shown match the scenario being evaluated.

The reported utilization is a **workload-to-capacity ratio**, calculated as total processing time divided by the number of available berths times the planning horizon. It is not a solver objective and is not necessarily the fraction of time actually occupied within the horizon; in a constrained scenario, scheduled work may extend beyond the horizon and the ratio can exceed 100%. The results describe this heuristic and these assumptions, not the optimal response to changing berth capacity.

In [ ]:
def analyze_berth_count_impact(base_ships, base_berths, planning_horizon):
    """Evaluate greedy schedule metrics for prefixes of the available berth list."""
    results = []

    for number_available in range(1, len(base_berths) + 1):
        available_berths = base_berths[:number_available]
        adjusted_ships = []

        for ship in base_ships:
            adjusted_ship = ship.copy()
            if adjusted_ship['preferred_berth'] > number_available:
                adjusted_ship['preferred_berth'] = number_available
            adjusted_ships.append(adjusted_ship)

        # Recompute costs because the preference adjustment defines this scenario's costs.
        available_cost_matrix = np.array([
            [calculate_assignment_cost(ship, berth) for berth in available_berths]
            for ship in adjusted_ships
        ])
        allocations, _ = simple_greedy_allocation(
            adjusted_ships,
            available_berths,
            available_cost_matrix
        )

        total_waiting = sum(allocation['waiting_time'] for allocation in allocations)
        total_cost = sum(allocation['cost'] for allocation in allocations)
        average_waiting = total_waiting / len(allocations)
        total_processing_time = sum(ship['processing_time'] for ship in adjusted_ships)
        total_berth_capacity = number_available * planning_horizon
        workload_ratio = total_processing_time / total_berth_capacity

        results.append({
            'num_berths': number_available,
            'total_waiting': total_waiting,
            'avg_waiting': average_waiting,
            'total_cost': total_cost,
            'workload_ratio': workload_ratio
        })

    return pd.DataFrame(results)


sensitivity_results = analyze_berth_count_impact(ships, berths, planning_horizon)
print("Greedy schedule sensitivity to available berth count:")
print(sensitivity_results.round(3))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
plot_specs = [
    ('total_waiting', 'Total waiting time (hours)', 'Total waiting time'),
    ('avg_waiting', 'Average waiting time (hours per ship)', 'Average waiting time'),
    ('total_cost', 'Total assignment cost', 'Assignment cost'),
    ('workload_ratio', 'Workload-to-capacity ratio (%)', 'Workload relative to capacity')
]

for axis, (column, y_label, title) in zip(axes.flat, plot_specs):
    values = sensitivity_results[column]
    if column == 'workload_ratio':
        values = values * 100
    axis.plot(
        sensitivity_results['num_berths'],
        values,
        marker='o',
        linewidth=2
    )
    axis.set_xlabel('Number of available berths')
    axis.set_ylabel(y_label)
    axis.set_title(title)
    axis.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print("\nGreedy schedule metrics by berth count:")
for _, row in sensitivity_results.iterrows():
    print(f"With {int(row['num_berths'])} available berth(s):")
    print(f"  Average waiting time: {row['avg_waiting']:.2f} hours per ship")
    print(f"  Workload-to-capacity ratio: {row['workload_ratio'] * 100:.1f}%")

## Key Takeaways and Limitations

### What the formulation and example demonstrate

- **Two kinds of decisions:** berth allocation is spatial (which berth) and temporal (when processing begins).
- **Feasibility requirements:** each ship receives one berth, service starts no earlier than arrival, and ships sharing a berth must not overlap.
- **Pairwise sequencing:** the binary precedence variable selects which ship comes first when a pair shares a berth; Big-$M$ terms make the ordering constraints conditional on that shared assignment.
- **Heuristic interpretation:** the executable schedule follows arrival order and uses preferred berths. It is a feasible example of scheduling, not an optimal MIP result.
- **Sensitivity interpretation:** changing the number of available berths changes the input scenario and the greedy schedule. The resulting metrics do not establish what an optimizer would achieve.

### Modeling limitations

- The mathematical formulation is not connected to an optimization solver in this notebook, so no MIP optimum, optimality gap, or solver runtime is reported.
- The greedy rule does not evaluate alternative berth assignments or globally reorder ships. It can therefore miss schedules with a better combined waiting-time and assignment-cost objective.
- Berth length, depth, and efficiency are recorded as descriptive data. The example's cost calculation uses berth length only for a simple size-related penalty; it does not enforce physical compatibility, use depth, or adjust processing duration by efficiency.
- The model and schedule are deterministic. They omit uncertain arrivals and processing times, setup times, tides, channel constraints, labor, and equipment availability.
- The MIP objective adds waiting time and assignment cost with equal numerical weight. In an operational application, the units and relative value of these terms must be specified consistently.

The central distinction is between a **mathematical optimization formulation**, which specifies the decisions and constraints, and a **solution procedure**, which actually computes a schedule. This notebook makes the formulation explicit and uses a transparent greedy procedure for its executable scheduling demonstration.